# Asahlagi — Train Quiz Generator (IndoT5 fine-tuning)

**Owner**: Audry (Backend — Quiz Generator)

This Colab notebook fine-tunes `Wikidepia/IndoT5-base` on the TyDiQA-id dataset for question generation, then pushes the result to Hugging Face Hub.

## Prerequisites

1. Colab runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU)
2. Hugging Face account: [huggingface.co/join](https://huggingface.co/join)
3. HF Access Token (Write permissions): https://huggingface.co/settings/tokens

## Expected runtime

- Setup: 5 min
- Training: 1-2 hours (3 epochs, ~5,500 samples)
- Push to Hub: 5 min

**Total**: ~2 hours of Colab time

## Reference

- [`/ML.md`](../../../../ML.md) §3 — DL strategy details
- [HuggingFace T5 docs](https://huggingface.co/docs/transformers/model_doc/t5)
- [TyDiQA dataset](https://huggingface.co/datasets/tydiqa)

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.2 datasets==3.0.1 accelerate==0.34.2 sentencepiece==0.2.0 evaluate==0.4.3 sacrebleu==2.4.3

## 2. Login to Hugging Face Hub

Paste your HF access token (from https://huggingface.co/settings/tokens). Make sure it has **Write** permissions.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 3. Load TyDiQA-id dataset

TyDiQA is multi-language. We filter to Indonesian only.

In [ ]:
from datasets import load_dataset

raw = load_dataset("tydiqa", "secondary_task")

def is_indonesian(example):
    return example["id"].startswith("indonesian")

ds_train = raw["train"].filter(is_indonesian)
ds_val = raw["validation"].filter(is_indonesian)

print(f"Train: {len(ds_train)} samples")
print(f"Val:   {len(ds_val)} samples")
print(f"\nSample:")
print(ds_train[0])

## 4. Load tokenizer and base model

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL_BASE = "Wikidepia/IndoT5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_BASE)
model = T5ForConditionalGeneration.from_pretrained(MODEL_BASE)

print(f"Loaded {MODEL_BASE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 5. Preprocess dataset for QG

Convert (passage, question) pairs into the format the T5 model expects:
- Input: `"buat pertanyaan: <context>"`
- Output: `<question>`

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64

def preprocess(examples):
    inputs = [f"buat pertanyaan: {ctx}" for ctx in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)

    labels = tokenizer(
        text_target=examples["question"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
tokenized_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

print(f"Tokenized train: {len(tokenized_train)} samples")

## 6. Set up training arguments

Hyperparameters tuned for T4 GPU memory and ~1-2 hour training budget.

In [ ]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = "/content/indot5-quizgen-asahlagi"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,            # mixed precision — fits T4 memory
    logging_steps=50,
    save_strategy="epoch",
    push_to_hub=False,    # we push manually at the end
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

## 7. Train

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## 8. Evaluate — sample generation

Manually inspect 20-30 generated questions for quality before publishing.

In [ ]:
model.eval()

test_passages = [
    "Fotosintesis adalah proses pembentukan glukosa oleh tumbuhan hijau dengan bantuan cahaya matahari dan klorofil. Proses ini terjadi di kloroplas dan menghasilkan oksigen sebagai produk samping.",
    "Algoritma adalah serangkaian langkah-langkah logis yang sistematis untuk menyelesaikan suatu masalah. Algoritma harus memiliki input, proses, dan output yang jelas.",
    "Inflasi adalah kenaikan harga barang dan jasa secara umum dan terus-menerus dalam jangka waktu tertentu. Inflasi dapat disebabkan oleh peningkatan permintaan atau penurunan penawaran.",
]

for passage in test_passages:
    prompt = f"buat pertanyaan: {passage}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)
    outputs = model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPassage: {passage[:80]}...")
    print(f"Generated: {question}")

## 9. Push to Hugging Face Hub

**Edit** the `HF_USERNAME` below to your HF username.

In [ ]:
HF_USERNAME = "audry-asahlagi"     # ← REPLACE with your HF username
MODEL_REPO = "indot5-quizgen-asahlagi"

full_repo = f"{HF_USERNAME}/{MODEL_REPO}"

model.push_to_hub(full_repo)
tokenizer.push_to_hub(full_repo)

print(f"\n✓ Pushed model to: https://huggingface.co/{full_repo}")
print("\nNext steps:")
print(f"  1. Verify model is accessible: open https://huggingface.co/{full_repo}")
print(f"  2. Update backend/ml/generator/inference.py:")
print(f"     _MODEL_NAME = \"{full_repo}\"")
print(f"  3. Test in backend: python -m ml.generator.inference")